# 01 — Data audit: Freddie Mac loan-level data

**Goal:** Check that the 2015 to 2019 files are complete and can be joined, learn what each code means, and settle the default definition and sample size before building anything.

## Setup

The raw files are pipe-delimited text with no header row. Column names come from the header files. DuckDB queries the files on disk, so nothing is loaded into memory until a query returns. Every column is read as text so that codes like `RA` or `9999` are not lost before we look at them.

In [ ]:
import re
from pathlib import Path

import duckdb
import yaml

ROOT = Path.cwd().parent
CONFIG_DIR = ROOT / "configs"
cfg = yaml.safe_load((CONFIG_DIR / "default.yaml").read_text())
RAW = (CONFIG_DIR / cfg["data"]["raw_dir"]).resolve()
INTERIM = (CONFIG_DIR / cfg["data"]["interim_dir"]).resolve()

# Set to a quarter like "2015Q1" while developing, or None for all 2015-2019 files.
DEV_QUARTER = None


def read_header(name: str) -> list[str]:
    """Read a pipe-delimited header file and return snake_case column names."""
    raw = (RAW / "header" / name).read_text().strip().split("|")
    return [re.sub(r"[^a-z0-9]+", "_", c.lower()).strip("_") for c in raw]


ORIG_COLS = read_header("origination_data_file_header.txt")
PERF_COLS = read_header("performance_data_file_header.txt")
len(ORIG_COLS), len(PERF_COLS)

In [ ]:
con = duckdb.connect()
con.execute("SET memory_limit = '8GB'")
con.execute("SET threads = 8")

quarter = DEV_QUARTER or "*"


def create_view(name: str, prefix: str, cols: list[str]) -> None:
    """Create a view over the raw text files. All columns are read as text."""
    pattern = (
        RAW
        / "historical_data_*"
        / f"historical_data_{quarter}"
        / f"{prefix}_{quarter}.txt"
    )
    con.execute(f"""
        CREATE OR REPLACE VIEW {name} AS
        SELECT
            regexp_extract(filename, '(\\d{{4}}Q\\d)', 1) AS quarter,
            * EXCLUDE (filename)
        FROM read_csv(
            '{pattern}',
            delim = '|',
            header = false,
            names = {cols},
            all_varchar = true,
            filename = true
        )
    """)


create_view("orig", "orig", ORIG_COLS)
create_view("perf", "perf", PERF_COLS)
con.sql("SELECT * FROM orig LIMIT 5").df()

In [ ]:
import pandas as pd


def pct(cond: str) -> str:
    """Return SQL for the percent of rows where `cond` is true. NULL counts as false."""
    return f"100.0 * avg(CASE WHEN {cond} THEN 1 ELSE 0 END)"


def show(sql: str) -> pd.DataFrame:
    """Run a query and return a rounded DataFrame."""
    return con.sql(sql).df().round(2)


CREDIT_EXIT = "('02', '03', '09')"
D90 = "(TRY_CAST(dlq AS INT) >= 3 OR dlq = 'RA')"
D180 = "(TRY_CAST(dlq AS INT) >= 6 OR dlq = 'RA')"
IN_RELIEF = "(borrower_assistance_plan = 'F' OR delinquency_due_to_disaster = 'Y')"

## Slim cache of the performance files

Every query on the raw performance text scans 45 GB. This writes the columns the audit needs to one Parquet file (about 1.8 GB) the first time it runs. It takes about 10 minutes. With `DEV_QUARTER` set, it skips the cache and reads the raw files for that quarter.

In [ ]:
SLIM = """
    SELECT
        quarter,
        loan_identifier,
        CAST(period AS INTEGER) AS period,
        CAST(loan_age AS INTEGER) AS loan_age,
        CAST(current_actual_upb AS DOUBLE) AS current_actual_upb,
        current_loan_delinquency_status AS dlq,
        modification_flag,
        zero_balance_code,
        zero_balance_effective_date,
        payment_deferral_flag,
        delinquency_due_to_disaster,
        borrower_assistance_plan,
        CAST(zero_balance_removal_upb AS DOUBLE) AS zero_balance_removal_upb,
        CAST(actual_loss AS DOUBLE) AS actual_loss
    FROM perf
"""

cache = INTERIM / "perf_slim.parquet"
if DEV_QUARTER:
    source = f"({SLIM})"
else:
    if not cache.exists():
        INTERIM.mkdir(parents=True, exist_ok=True)
        con.execute(f"COPY ({SLIM}) TO '{cache}' (FORMAT parquet, COMPRESSION zstd)")
    source = f"'{cache}'"

con.execute(f"""
    CREATE OR REPLACE VIEW p AS
    SELECT *, left(quarter, 4) AS vintage FROM {source}
""")

## Row counts

Loans and loan-months per vintage.

In [ ]:
show("""
    SELECT vintage, count(*) AS loan_months, count(DISTINCT loan_identifier) AS loans,
        min(period) AS first_period, max(period) AS last_period
    FROM p GROUP BY 1 ORDER BY 1
""")

## Join integrity

Orphan loans in either file, duplicate `(loan_identifier, period)` rows, and gaps in a loan's monthly history.

In [ ]:
show("""
    SELECT
        (SELECT count(*) - count(DISTINCT loan_identifier) FROM orig) AS dup_orig_ids,
        (SELECT count(*) FROM (SELECT loan_identifier, period FROM p
            GROUP BY 1, 2 HAVING count(*) > 1)) AS dup_loan_months,
        (SELECT count(*) FROM (SELECT DISTINCT loan_identifier FROM p)
            ANTI JOIN orig USING (loan_identifier)) AS perf_without_orig,
        (SELECT count(*) FROM orig
            ANTI JOIN (SELECT DISTINCT loan_identifier FROM p) USING (loan_identifier))
            AS orig_without_perf
""")

A loan has a gap if its number of rows is smaller than the number of months between its first and last row. This needs a one-row-per-loan summary, which the later sections also use.

In [ ]:
con.execute(f"""
    CREATE OR REPLACE TABLE loans AS
    SELECT
        loan_identifier,
        any_value(vintage) AS vintage,
        count(*) AS n_months,
        min((period // 100) * 12 + period % 100) AS first_month,
        max((period // 100) * 12 + period % 100) AS last_month,
        max(loan_age) AS max_age,
        max(zero_balance_code) AS zbc,
        CAST(max(zero_balance_effective_date) AS INTEGER) AS zbc_date,
        min(period) FILTER (WHERE {D90}) AS first_d90,
        min(period) FILTER (WHERE {D90} AND NOT coalesce({IN_RELIEF}, false))
            AS first_d90_no_relief,
        min(period) FILTER (WHERE {D180}) AS first_d180,
        max(period) FILTER (WHERE TRY_CAST(dlq AS INT) = 0) AS last_current,
        coalesce(bool_or(borrower_assistance_plan = 'F'), false) AS ever_forbearance,
        coalesce(bool_or(modification_flag IN ('Y', 'P')), false) AS ever_modified
    FROM p
    GROUP BY 1
""")
show("SELECT count(*) AS loans FROM loans")

In [ ]:
show("""
    SELECT count(*) AS loans_with_gaps FROM loans
    WHERE n_months <> last_month - first_month + 1
""")

## Codes and missing values

Codes for "not available" come from the Freddie Mac General User Guide. Percent of loans with each field missing:

In [ ]:
SENTINELS = {
    "classic_fico": "9999",
    "vantagescore_4_0": "9999",
    "original_debt_to_income_dti_ratio": "999",
    "original_loan_to_value_ltv": "999",
    "original_combined_loan_to_value_cltv": "999",
    "mortgage_insurance_percentage_mi": "999",
    "first_time_homebuyer_indicator": "9",
    "number_of_units": "99",
    "occupancy_status": "9",
    "channel": "9",
    "property_type": "99",
    "loan_purpose": "9",
    "number_of_borrowers": "99",
    "property_valuation_method": "7",
}
parts = [f"{pct(f'{col} = ' + repr(code))} AS {col}" for col, code in SENTINELS.items()]
msa = "metropolitan_statistical_area_msa_or_metropolitan_division"
parts.append(f"{pct(f'{msa} IS NULL')} AS msa")
show(f"SELECT {', '.join(parts)} FROM orig").T

Fields with a single value carry no information.

In [ ]:
for col in [
    "amortization_type",
    "interest_only_i_o_indicator",
    "prepayment_penalty_indicator",
]:
    print(show(f"SELECT {col}, count(*) AS n FROM orig GROUP BY 1"), "\n")

Code values in the performance file.

In [ ]:
for col in [
    "dlq",
    "zero_balance_code",
    "borrower_assistance_plan",
    "payment_deferral_flag",
    "delinquency_due_to_disaster",
    "modification_flag",
]:
    print(
        show(f"SELECT {col}, count(*) AS n FROM p GROUP BY 1 ORDER BY 2 DESC LIMIT 12"),
        "\n",
    )

## How loans end

`zero_balance_code`: 01 is prepaid or matured, 02, 03 and 09 are credit losses (third party sale, short sale, foreclosure sale), 15 and 16 are loans Freddie Mac sold, 96 is a defect. No code means the loan is still active (censored).

In [ ]:
show(f"""
    SELECT vintage, count(*) AS loans,
        {pct("zbc IS NULL")} AS active,
        {pct("zbc = '01'")} AS prepaid,
        {pct(f"zbc IN {CREDIT_EXIT}")} AS credit_loss,
        {pct("zbc IN ('15', '16')")} AS sold,
        {pct("zbc = '96'")} AS defect
    FROM loans GROUP BY 1 ORDER BY 1
""")

How many prepaid loans actually reached maturity?

In [ ]:
show("""
    SELECT count(*) AS prepaid_or_matured,
        count(*) FILTER (
            WHERE max_age >= CAST(o.original_loan_term AS INT) - 1
        ) AS matured
    FROM loans JOIN orig o USING (loan_identifier) WHERE zbc = '01'
""")

Loss severity on credit-loss exits. `actual_loss` is in dollars and `zero_balance_removal_upb` is the balance at the exit.

In [ ]:
show("""
    SELECT zero_balance_code, count(*) AS n, count(actual_loss) AS has_loss,
        median(actual_loss) AS median_loss,
        100 * median(actual_loss / nullif(zero_balance_removal_upb, 0))
            AS median_severity_pct
    FROM p WHERE zero_balance_code IN ('02', '03', '09', '15', '16')
    GROUP BY 1 ORDER BY 1
""")

## Delinquency

Delinquency codes: 0 is current, 1 is 30 to 59 days late, 2 is 60 to 89, 3 is 90 to 119, and so on. `RA` means the lender took the property (REO). How many loans ever reach 90+ days late, and what happens to them?

In [ ]:
show(f"""
    SELECT vintage,
        {pct("first_d90 IS NOT NULL")} AS ever_d90,
        {pct("first_d180 IS NOT NULL")} AS ever_d180,
        {pct("ever_forbearance")} AS ever_forbearance,
        {pct("ever_modified")} AS ever_modified
    FROM loans GROUP BY 1 ORDER BY 1
""")

In [ ]:
show(f"""
    SELECT count(*) AS loans_ever_d90,
        {pct("last_current > first_d90")} AS current_again_later,
        {pct(f"zbc IN {CREDIT_EXIT}")} AS credit_loss,
        {pct("zbc = '01'")} AS prepaid,
        {pct("zbc IS NULL")} AS still_active
    FROM loans WHERE first_d90 IS NOT NULL
""")

## COVID

Share of active loans 90+ days late, in forbearance, or flagged as disaster related, every quarter from 2019 to 2022.

In [ ]:
show(f"""
    SELECT period, count(*) AS active_loans,
        {pct(D90)} AS d90,
        {pct("borrower_assistance_plan = 'F'")} AS forbearance,
        {pct("delinquency_due_to_disaster = 'Y'")} AS disaster,
        {pct("payment_deferral_flag = 'C'")} AS deferral_this_month
    FROM p WHERE period BETWEEN 201901 AND 202212 AND period % 3 = 0
    GROUP BY 1 ORDER BY 1
""")

Share of 90+ day late months that are in forbearance or disaster relief, before and during COVID.

In [ ]:
show(f"""
    SELECT
        CASE WHEN period < 202003 THEN 'before Mar 2020'
            ELSE 'Apr 2020 to Dec 2021' END AS window,
        count(*) AS d90_months, {pct(IN_RELIEF)} AS in_relief
    FROM p WHERE {D90} AND period <= 202112 GROUP BY 1
""")

## One loan by hand

A 2016 loan that ended in a foreclosure sale (code 09). It reached 90 days late in Feb 2017, caught up in Mar 2017, fell behind again, and the lender took the property in Jan 2018.

In [ ]:
show("""
    SELECT period, loan_age, current_actual_upb, dlq, borrower_assistance_plan,
        zero_balance_code, zero_balance_effective_date, actual_loss
    FROM p WHERE loan_identifier = 'F16Q10002018' ORDER BY period
""")

## Default definition

Candidate rules. In each, the default date is the first month the rule is met.

- **A:** 90+ days late, or a credit-loss exit.
- **B:** Same as A, but 90+ day late months in forbearance or disaster relief do not count.
- **C:** 180+ days late, or a credit-loss exit.
- **D:** Credit-loss exit only.

Sold (15, 16) and defect (96) exits are censored in all four.

In [ ]:
EXIT_DATE = f"CASE WHEN zbc IN {CREDIT_EXIT} THEN zbc_date END"
RULES = {
    "A": f"least(first_d90, {EXIT_DATE})",
    "B": f"least(first_d90_no_relief, {EXIT_DATE})",
    "C": f"least(first_d180, {EXIT_DATE})",
    "D": EXIT_DATE,
}
cols = ", ".join(f"count({rule}) AS rule_{name}" for name, rule in RULES.items())
show(f"SELECT vintage, count(*) AS loans, {cols} FROM loans GROUP BY 1 ORDER BY 1")

## Events per vintage

The split censors training outcomes when validation starts (Jan 2018) and validation outcomes when test starts (Jan 2019). So what matters is not how many defaults a vintage has in total, but when they happen. Defaults under rule B by vintage and calendar year:

In [ ]:
show(f"""
    PIVOT (SELECT vintage, {RULES["B"]} // 100 AS year FROM loans
           WHERE {RULES["B"]} IS NOT NULL)
    ON year USING count(*) GROUP BY vintage ORDER BY vintage
""")

The same for rule A. The 2020 column shows the COVID forbearance spike.

In [ ]:
show(f"""
    PIVOT (SELECT vintage, {RULES["A"]} // 100 AS year FROM loans
           WHERE {RULES["A"]} IS NOT NULL)
    ON year USING count(*) GROUP BY vintage ORDER BY vintage
""")

Prepayments by vintage and calendar year.

In [ ]:
show("""
    PIVOT (SELECT vintage, zbc_date // 100 AS year FROM loans WHERE zbc = '01')
    ON year USING count(*) GROUP BY vintage ORDER BY vintage
""")

Defaults left after censoring, under rule B, for a few loan sample rates.

In [ ]:
B = RULES["B"]
B = RULES["B"]
usable = show(f"""
    SELECT
        count(*) FILTER (WHERE vintage <= '2017' AND {B} < 201801) AS train,
        count(*) FILTER (WHERE vintage = '2018' AND {B} < 201901) AS validation,
        count(*) FILTER (WHERE vintage = '2019' AND {B} IS NOT NULL) AS test
    FROM loans
""")
pd.DataFrame(
    {f"{r:.0%} sample": (usable.iloc[0] * r).round() for r in [0.02, 0.05, 0.10, 0.20]}
)

## Origination features

Medians and shares by vintage, with "not available" codes removed.

In [ ]:
FICO = "TRY_CAST(NULLIF(classic_fico, '9999') AS INT)"
LTV = "TRY_CAST(NULLIF(original_loan_to_value_ltv, '999') AS INT)"
DTI = "TRY_CAST(NULLIF(original_debt_to_income_dti_ratio, '999') AS INT)"
show(f"""
    SELECT left(quarter, 4) AS vintage,
        median({FICO}) AS fico_median,
        median({LTV}) AS ltv_median,
        {pct(f"{LTV} > 80")} AS ltv_over_80,
        median({DTI}) AS dti_median,
        {pct(f"{DTI} > 43")} AS dti_over_43,
        avg(CAST(original_interest_rate AS DOUBLE)) AS rate_mean,
        {pct("loan_purpose = 'P'")} AS purchase,
        {pct("loan_purpose = 'C'")} AS cash_out,
        {pct("original_loan_term = '360'")} AS term_30y,
        {pct("occupancy_status = 'I'")} AS investor,
        {pct("harp_indicator = 'Y'")} AS harp
    FROM orig GROUP BY 1 ORDER BY 1
""")

HARP loans (relief refinances of underwater loans) compared with the rest, under rule B.

In [ ]:
show(f"""
    SELECT o.harp_indicator, count(*) AS loans,
        {pct(RULES["B"] + " IS NOT NULL")} AS default_b,
        {pct("zbc = '01'")} AS prepaid
    FROM loans JOIN orig o USING (loan_identifier) GROUP BY 1
""")

## Findings

1. **The files are complete and join cleanly.** 7.61 million loans and 416 million loan-months across 2015 to 2019. No duplicate loans or loan-months, every loan appears in both files, and only 86 loans have a gap in their monthly history. Performance runs through Mar 2026.

2. **Some fields are empty or constant.** VantageScore is missing for every loan and property valuation method for 88% to 100% (it starts in 2017). Every loan is fixed rate with no interest-only period and no prepayment penalty. Drop these. DTI is missing for 3.5% of loans, always for HARP loans. MSA is blank for 10%. Credit score and LTV are almost never missing.

3. **The performance codes do not all match the guide.** Delinquency comes as two-digit text (`00`, `01`, ...) plus `RA`. The payment deferral flag uses `C` for the current month, not `Y`. Read every column as text first and convert on purpose.

4. **Most loans prepay. Very few end in a credit loss.** By Mar 2026, 70% to 80% of each vintage has prepaid and 19% to 30% is still active. Only 0.07% to 0.19% ended in a credit-loss exit, and another 0.1% to 0.2% were sold by Freddie Mac. Only 1,470 of the 5.6 million `01` exits are true maturities, so `01` can be treated as prepayment.

5. **Being 90+ days late is not the same as losing money.** 4.0% to 5.7% of each vintage was 90+ days late at some point, but 95% of those loans were current again at some point, and only 2.7% ended in a credit loss. 48% later prepaid and 45% are still active.

6. **COVID drives most 90+ day late events.** The share of active loans 90+ days late went from 0.25% in Feb 2020 to 3.5% in Aug 2020 and stayed above 2% until late 2021. 89% of 90+ day late months from Apr 2020 to Dec 2021 were in forbearance or disaster relief, compared with 19% before COVID. Under rule A, 2020 alone has 252,000 first defaults, more than all other years together.

7. **The choice of rule changes the default count by a factor of 2 to 3.** Over 2015 to 2019 vintages, rule A gives 385,000 defaults, rule C 235,000, rule B 155,000 and rule D 10,000. Rule B removes the COVID spike but keeps the pre-COVID pattern.

8. **The censoring rule leaves few defaults for training and almost none for validation.** With training censored at Jan 2018, rule B leaves 12,300 training defaults in the full data, mostly from the 2015 vintage. Validation (2018 vintage, censored at Jan 2019) has only 521. A 5% sample would leave about 610 training, 26 validation and 1,570 test defaults. Prepayments are not a problem: there are about 438,000 training prepayments.

9. **The training data only covers the first 3 years of a loan.** Training loans are observed up to 36 months of age, while test loans are followed for up to 86 months. Hazards at older ages would be extrapolated.

10. **The vintages differ.** 2017 and 2018 have more purchase loans (58% and 68% vs 44%), more loans with LTV over 80 and DTI over 43, and higher rates (4.7% in 2018 vs 3.8% in 2016). 2019 looks more like 2015 and 2016. HARP loans (3.5% of loans, falling from 8.2% of the 2015 vintage to almost none in 2019) default at 4.9% under rule B, compared with 1.9% for other loans.

11. **Loss data exists for most credit-loss exits.** Median severity is about 27% of the balance for foreclosure sales (09), 15% for short sales (03) and 4% for third party sales (02). `actual_loss` is positive for a loss.

**Open choices for you:**

- Default rule (A, B, C or D), and how to treat COVID forbearance.
- Whether to drop HARP loans.
- Whether the split still works given findings 8 and 9. Options include an earlier validation start, censoring only training outcomes, or a longer training window.
- Loan sample rate.

**Decision:**

- Default rule B: 90+ days late or a credit-loss exit, not counting months in forbearance or disaster relief.
- Split: train on 2015 and 2016 loans, validate on 2017 loans, test on 2019 loans. Cutoff Jan 2019. 2018 loans are left out because they are less than a year old at the cutoff.
- Drop HARP loans.
- Sample 5% of loans.

See `docs/decisions.md` for the reasons.